In [12]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [13]:
# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

c:\Users\ADMIN\AppData\Local\Programs\Python\Python314\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [14]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

### Build the CNN

In [15]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2) ,

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening
        x = self.fc_layers(x)

        return x

In [16]:
model = CNN()

In [17]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [18]:
# epochs = 10

# for epoch in range(epochs):
#     epoch_training_loss = 0.0

#     for images, labels in trainloader:
#         optimizer.zero_grad()
        
#         output = model.forward(images) # FP
#         loss = criterion(output, labels) # loss fnx
#         loss.backward() # BP
#         optimizer.step() # update params

#         epoch_training_loss += loss.item()

#     print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

In [19]:
# # Evaludate our CNN

# correct_labels = 0
# total_labels = 0

# model.eval()

# with torch.no_grad():
#     for images, labels in testloader:
#         outputs = model.forward(images)
#         _, predicted  = torch.max(outputs, 1)

#         correct_labels += (predicted == labels).sum().item()
#         total_labels += labels.size(0)

# print(f"accuracy = {correct_labels / total_labels * 100}")

In [ ]:
epochs = 8

training_loss = []
validation_loss = []

best_epoch_loss = float("inf")

for epoch in range(epochs):

    model.train()
    epoch_training_loss = 0

    for images, labels in trainloader:

        optimizer.zero_grad()

        output = model(images)
        loss = criterion(output, labels)

        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    e_t_l = epoch_training_loss / len(trainloader)
    training_loss.append(e_t_l)

    model.eval()
    epoch_val_loss = 0

    with torch.no_grad():

        for xb, yb in testloader:

            output = model(xb)
            loss = criterion(output, yb)

            epoch_val_loss += loss.item()

    r_v_l = epoch_val_loss / len(testloader)
    validation_loss.append(r_v_l)

    print(f"Epoch {epoch+1}/{epochs}, Train Loss={e_t_l:.4f}, Val Loss={r_v_l:.4f}")

    if r_v_l < best_epoch_loss:
        best_epoch_loss = r_v_l
        torch.save(model.state_dict(), "great_model.pt")

Epoch 1/8, Train Loss=1.3691, Val Loss=1.0223
Epoch 2/8, Train Loss=0.9158, Val Loss=0.8426


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
data=pd.DataFrame({
    "training loss": training_loss,
    "test loss": validation_loss
})
plt.plot(df["training_loss"],hue=training_loss)
plt.plot(df["validation_loss"],hue=test_loss)
plt.xlabel("Epochs")
plt.ylabel("Losses")

plt.legend()


NameError: name 'df' is not defined

In [ ]:

model.load_state_dict(torch.load("best_model.pt"))